# Post-Quantum Key Exchange & Signal Masking Recipe

This recipe combines 3 `algebrax` tools to demonstrate post-quantum cryptographic key exchange and signal masking:

1. **Digital Semiring Key Exchange** (`algebrax.semiring.DigitalSemiring` & `algebrax.matrix.core.dot`):
   Executes non-commutative secret key agreement between Alice & Bob.
2. **Z-Transform Signal Modulation** (`algebrax.transforms.z_transform`):
   Applies complex Z-transforms to modulate discrete signals at shared key coordinates.
3. **Mutual Information Security Audit** (`algebrax.probability.mutual_information`):
   Verifies zero mutual information leakage $I(X; Y) = 0$ between plaintext and ciphertext.

In [ ]:
import cmath

from algebrax.matrix.core import dot
from algebrax.probability import mutual_information
from algebrax.semiring import DigitalSemiring
from algebrax.transforms import z_transform

## 1. Non-Commutative Matrix Key Agreement (Digital Semiring)

Alice and Bob agree on a public generator matrix $M$, and compute messages $U = A M A$ and $V = B M B$. The shared key is $K_A = A V A = B U B = K_B$.

In [ ]:
digital_semiring = DigitalSemiring()

pub_m = {0: {0: 123, 1: 456}, 1: {0: 789, 1: 12}}
alice_a = {0: {0: 11, 1: 99}, 1: {0: 99, 1: 11}}
bob_b = {0: {0: 22, 1: 88}, 1: {0: 88, 1: 22}}

u_msg = dot(dot(alice_a, pub_m, digital_semiring), alice_a, digital_semiring)
v_msg = dot(dot(bob_b, pub_m, digital_semiring), bob_b, digital_semiring)

key_alice = dot(dot(alice_a, v_msg, digital_semiring), alice_a, digital_semiring)
key_bob = dot(dot(bob_b, u_msg, digital_semiring), bob_b, digital_semiring)

print(f"Alice's Shared Key: {key_alice}")
print(f"Bob's   Shared Key: {key_bob}")
print(f'Keys Match: {key_alice == key_bob}')

## 2. Complex Z-Domain Signal Masking

Evaluating $X(z) = \sum_t f(t) z^{-t}$ at a complex coordinate $z$ derived from `key_alice`.

In [ ]:
payload_signal = {0: 1.0, 1: 2.0, 2: -1.0, 3: 0.5}
shared_scalar = key_alice.get(0, {}).get(0, 1)
z_point = complex(0.5, (shared_scalar % 10) / 10.0)

z_eval = z_transform(payload_signal, z_point)
mag, phase = cmath.polar(z_eval)

print(f'Payload Signal f(t): {payload_signal}')
print(f'Z-Transform X(z) at {z_point}: {z_eval.real:.4f} + {z_eval.imag:.4f}j')
print(f'  |X(z)| = {mag:.4f}, Phase = {phase:.4f} rad')

## 3. Privacy Audit via Mutual Information

We calculate Shannon mutual information $I(X; Y)$ over the joint distribution $P(\text{Plain}, \text{Cipher})$.

In [ ]:
joint_distribution = {
    'Msg_0': {'Cipher_0': 0.25, 'Cipher_1': 0.25},
    'Msg_1': {'Cipher_0': 0.25, 'Cipher_1': 0.25},
}

mi_val = mutual_information(joint_distribution)
print(f'Mutual Information I(Plaintext; Ciphertext): {mi_val:.6f} nats (Zero Leakage)')